In [0]:
# Instala as bibliotecas necessárias
# Com Spark, precisa de menos bibliotecas
%pip install python-dotenv

In [0]:
# Imports e Configuração

import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path="/Workspace/Repos/luizhpdatasci@gmail.com/merca-data-platform/.env")

print("✅ Imports carregados com sucesso!")

In [0]:
# Função Conexão SQL via JDBC

def get_jdbc_config():
    host     = os.getenv("jdbc_hostname")
    database = os.getenv("jdbc_database")
    username = os.getenv("jdbc_username")
    password = os.getenv("jdbc_password")

    if not all([host, database, username, password]):
        raise ValueError("❌ Uma ou mais variáveis SQL do .env estão vazias.")

    jdbc_url = f"jdbc:sqlserver://{host}:1433;database={database};encrypt=true;trustServerCertificate=false;"

    properties = {
        "user"    : username,
        "password": password,
        "driver"  : "com.microsoft.sqlserver.jdbc.SQLServerDriver"
    }

    return jdbc_url, properties

jdbc_url, jdbc_properties = get_jdbc_config()
print("✅ Configuração JDBC criada!")

In [0]:
# Configurar Acesso ao ADLS via Spark

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from io import BytesIO

def configurar_acesso_adls():
    storage_account   = os.getenv("storage_account_name")
    client_id_val     = os.getenv("client_id")
    tenant_id_val     = os.getenv("tenant_id")
    client_secret_val = os.getenv("client_secret")

    if not all([storage_account, client_id_val, tenant_id_val, client_secret_val]):
        raise ValueError("❌ Uma ou mais variáveis do .env estão vazias.")

    credential = ClientSecretCredential(
        tenant_id=tenant_id_val,
        client_id=client_id_val,
        client_secret=client_secret_val
    )

    account_url = f"https://{storage_account}.dfs.core.windows.net"
    client      = DataLakeServiceClient(
        account_url=account_url,
        credential=credential
    )

    print(f"✅ Acesso ao ADLS configurado para: {storage_account}")
    return client, storage_account

adls_client, storage_account = configurar_acesso_adls()
container = os.getenv("container_name")


## Acesso ao Data Lake
"""
O acesso ao ADLS Gen2 foi implementado via SDK Azure 
(azure-storage-file-datalake) com autenticação por 
Service Principal, substituindo o mount tradicional 
(dbutils.fs.mount), que não é suportado em clusters 
Serverless e está deprecado nas versões mais recentes 
do Databricks com Unity Catalog.
"""


In [0]:
# Listar Arquivos do Container

def listar_arquivos(adls_client, container):
    fs_client = adls_client.get_file_system_client(file_system=container)
    arquivos  = [p.name for p in fs_client.get_paths() if not p.is_directory]

    print(f"📁 Arquivos encontrados: {len(arquivos)}\n")
    for arquivo in arquivos:
        print(f"   - {arquivo}")

    return arquivos

arquivos = listar_arquivos(adls_client, container)

In [0]:
# Configuração JDBC SQL Server

def get_sql_options():
    host     = os.getenv("jdbc_hostname")
    database = os.getenv("jdbc_database")
    username = os.getenv("jdbc_username")
    password = os.getenv("jdbc_password")

    if not all([host, database, username, password]):
        raise ValueError("❌ Uma ou mais variáveis SQL do .env estão vazias.")

    options = {
        "host"                   : host,
        "port"                   : "1433",
        "database"               : database,
        "user"                   : username,
        "password"               : password,
        "encrypt"                : "true",
        "trustServerCertificate" : "false"
    }

    print("✅ Configuração SQL Server criada!")
    return options

sql_options = get_sql_options()

In [0]:
# Teste JDBC (CREATE/INSERT)

def testar_conexao_sql():
    options = get_sql_options()

    # Cria DataFrame de teste
    df_teste = spark.createDataFrame(
        [(1, "Conexão com sucesso")],
        ["id", "mensagem"]
    )

    # Testa escrita
    df_teste.write \
        .format("sqlserver") \
        .option("host", options["host"]) \
        .option("port", options["port"]) \
        .option("database", options["database"]) \
        .option("user", options["user"]) \
        .option("password", options["password"]) \
        .option("dbtable", "squad3.teste_conexao") \
        .option("encrypt", options["encrypt"]) \
        .option("trustServerCertificate", options["trustServerCertificate"]) \
        .mode("overwrite") \
        .save()

    print("✅ Conexão SQL Server testada com sucesso!")
    print(f"   Permissão de CREATE : ✅")
    print(f"   Permissão de INSERT : ✅")

    # Lê de volta para confirmar
    df_verificacao = spark.read \
        .format("sqlserver") \
        .option("host", options["host"]) \
        .option("port", options["port"]) \
        .option("database", options["database"]) \
        .option("user", options["user"]) \
        .option("password", options["password"]) \
        .option("dbtable", "squad3.teste_conexao") \
        .option("encrypt", options["encrypt"]) \
        .option("trustServerCertificate", options["trustServerCertificate"]) \
        .load()

    print(f"   Registro inserido   : ✅")
    df_verificacao.show()

testar_conexao_sql()

In [0]:
# Função Ler parquet do Data Lake

def ler_parquet(adls_client, container, caminho_arquivo):
    fs_client   = adls_client.get_file_system_client(file_system=container)
    file_client = fs_client.get_file_client(caminho_arquivo)
    conteudo    = file_client.download_file().readall()
    return pd.read_parquet(BytesIO(conteudo))

print("✅ Função ler_parquet criadas!")

In [0]:
# Ler Arquivo do Data Lake

df = ler_parquet(adls_client, container,"vendas_raw/2026/02/21/112200/ecommerce_pedidos.parquet")   

print(f"✅ Arquivo lido com sucesso! Shape: {df.shape}")

In [0]:
# Análise Exploratória

print("=" * 50)
print("📊 ANÁLISE EXPLORATÓRIA")
print("=" * 50)

print(f"\n📐 Shape: {df.shape}")
print(f"   {df.shape[0]} linhas | {df.shape[1]} colunas")

print("\n📋 Colunas e tipos:")
print(df.dtypes)

print("\n❓ Nulos por coluna:")
print(df.isnull().sum())

print(f"\n🔁 Duplicatas: {df.duplicated().sum()}")

print("\n📈 Estatísticas:")
df.describe()

In [0]:
# Primeiras linhas

print("👀 Primeiras 10 linhas:")
df.head(10)

In [0]:
# Tratamento básico de dados 

print("=" * 50)
print("🔧 TRATAMENTOS — ecommerce_pedidos")
print("=" * 50)

# Datas
colunas_data = ['dt_pedido', 'dt_ultima_atualizacao_status']
for col in colunas_data:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f"✅ {col} convertida para datetime")

# Duplicatas
total_antes = len(df)
df = df.drop_duplicates(subset=['id_pedido'])
print(f"\n✅ Duplicatas removidas: {total_antes - len(df)} linhas")

# Nulos
df['status_pedido']    = df['status_pedido'].fillna('desconhecido')
df['metodo_pagamento'] = df['metodo_pagamento'].fillna('desconhecido')
df['valor_total']      = df['valor_total'].fillna(0)
df['valor_frete']      = df['valor_frete'].fillna(0)
print("\n✅ Nulos tratados")

# Padronizar texto
df['status_pedido']    = df['status_pedido'].str.strip().str.upper()
df['metodo_pagamento'] = df['metodo_pagamento'].str.strip().str.lower()
print("\n✅ Texto padronizado")

# Tipos
df['id_pedido']           = df['id_pedido'].astype(str)
df['id_cliente']          = df['id_cliente'].astype(str)
df['id_endereco_entrega'] = df['id_endereco_entrega'].astype(str)
df['valor_total']         = df['valor_total'].astype(float)
df['valor_frete']         = df['valor_frete'].astype(float)
print("\n✅ Tipos garantidos")

print(f"\n📐 Shape final: {df.shape}")
df.head()

In [0]:
# Salvar no SQL Server

def salvar_tabela(df, nome_tabela, modo="overwrite"):
    options = get_sql_options()

    # Converte pandas → Spark
    df_spark = spark.createDataFrame(df)

    df_spark.write \
        .format("sqlserver") \
        .option("host", options["host"]) \
        .option("port", options["port"]) \
        .option("database", options["database"]) \
        .option("user", options["user"]) \
        .option("password", options["password"]) \
        .option("dbtable", f"squad3.{nome_tabela}") \
        .option("encrypt", options["encrypt"]) \
        .option("trustServerCertificate", options["trustServerCertificate"]) \
        .mode(modo) \
        .save()

    print(f"✅ Tabela squad3.{nome_tabela} salva! ({len(df)} linhas)")

salvar_tabela(df, nome_tabela="ecommerce_pedidos")

In [0]:
# Verificar Dados Salvos

def consultar_tabela(nome_tabela):
    options = get_sql_options()

    df_resultado = spark.read \
        .format("sqlserver") \
        .option("host", options["host"]) \
        .option("port", options["port"]) \
        .option("database", options["database"]) \
        .option("user", options["user"]) \
        .option("password", options["password"]) \
        .option("dbtable", f"squad3.{nome_tabela}") \
        .option("encrypt", options["encrypt"]) \
        .option("trustServerCertificate", options["trustServerCertificate"]) \
        .load()

    return df_resultado.toPandas()

df_verificacao = consultar_tabela("ecommerce_pedidos")

print(f"✅ Verificação concluída!")
print(f"   Linhas no banco: {len(df_verificacao)}")
df_verificacao.head()